# 01 · Carga y datos

Este notebook prepara los **dos únicos CSV base** que usa el resto del proyecto.
Se corre **una sola vez** (cuando llega el CSV grande de Kaggle) y deja en `data/`:

- `charts_ar_global.csv` — charts filtrados a Argentina + Global.
- `charts_argentina.csv` — subconjunto solo Argentina.

Las series por sub-pregunta (viral, catálogo, artistas AR/Global, intersección) **no**
se guardan como archivos: cada notebook (03, 04, 05) las arma en memoria con
`construir_serie()` + `interpolar_serie()` sobre estos dos CSV, en el momento en que
las necesita. Es rápido de recalcular y evita tener CSV sueltos.

## 1. De dónde bajar el dataset

Usamos **Spotify Charts** de Kaggle: `dhruvildave/spotify-charts`
(<https://www.kaggle.com/datasets/dhruvildave/spotify-charts>).

1. Crear una cuenta gratuita en Kaggle y entrar a la página del dataset.
2. Descargar `charts.csv` (**es enorme**: ~26 millones de filas, >1 GB
   descomprimido, con todos los países desde 2017).
3. Guardarlo como **`data/charts.csv`** (esa carpeta no se versiona, ver
   `.gitignore`).

Alternativa por línea de comandos (requiere la API de Kaggle configurada):

```bash
kaggle datasets download -d dhruvildave/spotify-charts -p data/ --unzip
```

Columnas del CSV: `title, rank, date, artist, url, region, chart, trend, streams`.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd

from src.utils import cargar_dataset_completo

## 2. `charts_ar_global.csv` — filtrar a Argentina + Global

`cargar_dataset_completo()` lee el archivo original **en chunks** (de a 500.000
filas) para no cargar 1 GB en memoria, y se queda solo con
`region ∈ {Argentina, Global}`.

In [ ]:
RUTA_CSV_GRANDE = "../data/charts.csv"            # <-- el archivo enorme de Kaggle
RUTA_AR_GLOBAL  = "../data/charts_ar_global.csv"  # <-- salida: Argentina + Global

df_ag = cargar_dataset_completo(RUTA_CSV_GRANDE)
df_ag.to_csv(RUTA_AR_GLOBAL, index=False)

print("charts_ar_global.csv")
print(f"    {len(df_ag):,} filas  |  regiones: {sorted(df_ag['region'].unique())}")
print(f"    {df_ag['date'].min().date()} → {df_ag['date'].max().date()}")

## 3. `charts_argentina.csv` — subconjunto solo Argentina

Se filtra directamente del general (no hace falta releer el CSV grande).

In [ ]:
RUTA_ARGENTINA = "../data/charts_argentina.csv"

df_ar = df_ag[df_ag["region"] == "Argentina"].sort_values("date").reset_index(drop=True)
df_ar.to_csv(RUTA_ARGENTINA, index=False)

print("charts_argentina.csv")
print(f"    {len(df_ar):,} filas  |  regiones: {sorted(df_ar['region'].unique())}")
print(f"    {df_ar['date'].min().date()} → {df_ar['date'].max().date()}")

## 4. Listo

Quedaron en `data/` (ninguno se versiona, están bajo `data/` ignorado por git):

| Archivo | Contenido | Lo usa |
|---|---|---|
| `charts_ar_global.csv` | charts de Argentina + Global | 04 y 05 |
| `charts_argentina.csv` | charts solo Argentina | 03 |

Desde acá, cada notebook arma sus propias series con `construir_serie()` +
`interpolar_serie()` sobre el CSV que le corresponda.